<a href="https://colab.research.google.com/github/kimanaai/myproject-/blob/main/%D0%9A%D0%BE%D0%BF%D0%B8%D1%8F_%D0%B1%D0%BB%D0%BE%D0%BA%D0%BD%D0%BE%D1%82%D0%B0_%22Aurora_Diamond_RAG_Groq_final_ipynb%22.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🕵️ Aurora Diamond Mystery — RAG with Groq (final version)

**Goal:** build a Retrieval-Augmented Generation (RAG) system from scratch, understand every layer, and then **tune** it like an engineer.

**Documents → Chunking → Embeddings → Chroma vector DB → Retrieval → Prompt + context → Groq LLM → Grounded answer → Evaluation**

### How this notebook is organised

| Part | What you do |
|---|---|
| **A. Build** | Build the full pipeline, step by step. Two bugs from the first version are fixed and explained. |
| **B. Tuning lab** | Change one setting at a time and measure what happens: chunk size, overlap, k, search type, embedding model, temperature, max tokens, LLM, and the prompt. |
| **C. Wrap-up** | Compare results, learn the difference between *tuning* and *fine-tuning*, and take on the challenges. |

### Before you start
1. Open the 🔑 **Secrets** panel in Colab.
2. Add a secret named `GROQ_API_KEY` (free key at console.groq.com).
3. Turn on **Notebook access** for that secret.
4. **Runtime → Run all** works, but you will learn more by running cells one at a time and **predicting the result before each run**.

> Embeddings run locally with `sentence-transformers`, so you need only one API key.

---
# Part A — Build the pipeline

### A0. Install libraries

In [1]:
!pip -q install \
    langchain \
    langchain-community \
    langchain-text-splitters \
    langchain-groq \
    langchain-chroma \
    chromadb \
    langchain-huggingface \
    sentence-transformers

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 36.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 77.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 16.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.5/7.5 MB 103.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 48.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 75.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.1/140.1 kB 9.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.3/60.3 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 206.3/206.3 kB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.7/95.7

### A1. Imports

In [2]:
import os, re, time, shutil
from pathlib import Path

import pandas as pd
import chromadb
from google.colab import userdata

from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_groq import ChatGroq

pd.set_option("display.max_colwidth", 120)

/tmp/ipykernel_1464/193365967.py:8: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import DirectoryLoader, TextLoader


### A2. Load the Groq API key

In [3]:
os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
print("Groq API key loaded:", bool(os.environ.get("GROQ_API_KEY")))

Groq API key loaded: True


### A3. The settings we will tune later

Every number below is a **design choice**, not a law of nature. In Part B you will change each one and measure the effect.

| Setting | What it controls | Pipeline stage |
|---|---|---|
| `CHUNK_SIZE` | Maximum characters in one chunk | Chunking |
| `CHUNK_OVERLAP` | Characters shared by neighbouring chunks | Chunking |
| `TOP_K` | How many chunks are sent to the LLM | Retrieval |
| `EMBEDDING_MODEL` | Which model turns text into vectors | Embedding |
| `LLM_MODEL` | Which model writes the answer | Generation |
| `TEMPERATURE` | How random the LLM's word choice is | Generation |
| `MAX_TOKENS` | Maximum length of the LLM's output | Generation |

In [4]:
CHUNK_SIZE      = 500
CHUNK_OVERLAP   = 50
TOP_K           = 5
EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
LLM_MODEL       = "openai/gpt-oss-120b"
TEMPERATURE     = 0
MAX_TOKENS      = 1500

## 1. Create the mystery case

We use several independent documents on purpose. The answer needs evidence from **different sources** (access logs, CCTV, interviews, policy), so the RAG system must *retrieve* the right pieces instead of reading one long text.

### The case
The **Aurora Diamond** disappeared from VIP Exhibition Room 204. It was confirmed present at **21:35** and found missing at **21:42**.

Suspects: **Alice Morgan**, **Bob Carter**, **Clara Lee**, **David Ross**.

In [5]:
DATA_DIR = Path("mystery_data")
if DATA_DIR.exists():
    shutil.rmtree(DATA_DIR)
DATA_DIR.mkdir()
print("Created:", DATA_DIR.resolve())

Created: /content/mystery_data


In [6]:
documents = {
    "incident_report.txt": '''
CASE ID: AD-204
Incident: Aurora Diamond disappearance

The Aurora Diamond was displayed in VIP Exhibition Room 204.
At 21:35, security officer Elena Park confirmed that the diamond was present in its display case.
At 21:42, Elena Park returned to the room and discovered that the diamond was missing.
The room showed no signs of forced entry.
The display case was not visibly damaged.

The incident was reported immediately to hotel security.
''',

    "access_logs.txt": '''
SECURITY ACCESS LOG — ROOM 204

21:37:12 — Badge B-17 — Bob Carter — ENTRY — Room 204
21:45:03 — Badge B-17 — Bob Carter — EXIT — Room 204

Security system notes:
Badge B-17 is assigned to Bob Carter.
No other badge was recorded entering Room 204 between 21:30 and 21:50.
''',

    "cctv_report.txt": '''
CCTV REVIEW — ROOM 204 CORRIDOR

21:36:58 — A person wearing a black hotel security uniform approaches Room 204.
21:37:14 — The person enters Room 204.
21:41:50 — The person remains inside the room area.
21:44:47 — The person leaves Room 204.
21:45:05 — The corridor is empty.

The camera angle does not provide a sufficiently clear facial image to identify the person with certainty.
The person appears to have the build and uniform style of a hotel security employee.
''',

    "bob_interview.txt": '''
INTERVIEW — BOB CARTER
Role: Hotel Security Officer

Bob Carter stated:
"I finished my work around 21:30 and left the hotel.
I did not return to Room 204 after that.
My security badge stayed with me the entire evening.
Nobody else had access to my badge."

Bob said he was not aware that the Aurora Diamond was missing until another officer informed him.
''',

    "alice_interview.txt": '''
INTERVIEW — ALICE MORGAN
Role: Event Coordinator

Alice Morgan stated that she was working in Room 201 during the relevant period.
She said she never entered Room 204.

Access records show that Alice Morgan did not have an authorized badge for Room 204.
''',

    "clara_interview.txt": '''
INTERVIEW — CLARA LEE
Role: Photographer

Clara Lee stated that she was photographing guests near Room 203.
Her camera contains timestamped photographs at approximately 21:36, 21:39, and 21:43.

Clara did not have an authorized security badge for Room 204.
''',

    "david_interview.txt": '''
INTERVIEW — DAVID ROSS
Role: Guest

David Ross stated that he was in the main hotel lobby between approximately 21:30 and 21:50.
Lobby camera footage places David in the lobby during the relevant period.

David did not have an authorized security badge for Room 204.
''',

    "security_policy.txt": '''
HOTEL SECURITY POLICY — VIP EXHIBITION ROOMS

1. After 21:30, only authorized hotel security badges may unlock VIP Exhibition Room 204.
2. Security badges must remain with the employee to whom they are assigned.
3. Every badge entry and exit is recorded by the electronic access system.
4. The employee assigned to a badge is responsible for access made using that badge unless evidence proves that the badge was used without authorization.
5. CCTV and electronic access records should be considered together when investigating incidents.
''',

    "timeline.txt": '''
CASE TIMELINE — AD-204

21:35 — Aurora Diamond confirmed present by security officer Elena Park.
21:37 — Badge B-17, assigned to Bob Carter, records entry into Room 204.
21:38 — CCTV shows a person in a black hotel security uniform inside Room 204.
21:42 — Aurora Diamond discovered missing.
21:44 — CCTV shows the person leaving Room 204.
21:45 — Badge B-17, assigned to Bob Carter, records exit from Room 204.

Important observation:
Bob Carter stated that he left the hotel at approximately 21:30 and did not return to Room 204.
The electronic access records contradict this statement unless the badge was used by someone else.
''',

}

for filename, content in documents.items():
    (DATA_DIR / filename).write_text(content.strip(), encoding="utf-8")

print(f"Created {len(documents)} evidence documents:")
for path in sorted(DATA_DIR.glob("*.txt")):
    print("  •", path.name)

Created 9 evidence documents:
  • access_logs.txt
  • alice_interview.txt
  • bob_interview.txt
  • cctv_report.txt
  • clara_interview.txt
  • david_interview.txt
  • incident_report.txt
  • security_policy.txt
  • timeline.txt


## 2. Load the documents (ingestion)

A production RAG system might ingest PDFs, web pages, tickets or a database. Plain text keeps every step visible.
Each loaded document keeps its **metadata**, including the file path in `source`. We need it later to cite sources.

In [7]:
loader = DirectoryLoader(
    str(DATA_DIR),
    glob="*.txt",
    loader_cls=TextLoader,
    loader_kwargs={"encoding": "utf-8"},
)
docs = loader.load()

def name(doc):
    # 'mystery_data/access_logs.txt' -> 'access_logs.txt'
    return Path(doc.metadata["source"]).name

print("Documents loaded:", len(docs), "\n")
for doc in docs:
    print(f"{name(doc):22} → {len(doc.page_content):4} characters")

Documents loaded: 9 

incident_report.txt    →  443 characters
access_logs.txt        →  271 characters
security_policy.txt    →  538 characters
alice_interview.txt    →  253 characters
timeline.txt           →  630 characters
bob_interview.txt      →  354 characters
david_interview.txt    →  266 characters
cctv_report.txt        →  469 characters
clara_interview.txt    →  256 characters


## 3. Chunking

The LLM does not receive the whole knowledge base. We cut documents into **chunks** so the retriever can hand over only the relevant pieces.

`RecursiveCharacterTextSplitter` tries to cut at the most natural place first:
1. paragraph breaks (`\n\n`), then
2. line breaks (`\n`), then
3. spaces, then
4. single characters (last resort).

It keeps adding pieces until the next one would exceed `chunk_size`. `chunk_overlap` repeats the end of one chunk at the start of the next, so a sentence cut at a boundary is not lost.

✍️ **Predict:** seven of our files are shorter than 500 characters. How many chunks will the nine files become?

In [8]:
def make_chunks(chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP):
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
    )
    return splitter.split_documents(docs)

chunks = make_chunks()
print("Documents:", len(docs), "→ chunks:", len(chunks), "\n")

for i, chunk in enumerate(chunks):
    text = chunk.page_content.replace("\n", " ")
    print(f"[{i:2}] {name(chunk):22} {len(chunk.page_content):3} ch | {text[:90]}")

Documents: 9 → chunks: 11 

[ 0] incident_report.txt    443 ch | CASE ID: AD-204 Incident: Aurora Diamond disappearance  The Aurora Diamond was displayed i
[ 1] access_logs.txt        271 ch | SECURITY ACCESS LOG — ROOM 204  21:37:12 — Badge B-17 — Bob Carter — ENTRY — Room 204 21:4
[ 2] security_policy.txt     44 ch | HOTEL SECURITY POLICY — VIP EXHIBITION ROOMS
[ 3] security_policy.txt    492 ch | 1. After 21:30, only authorized hotel security badges may unlock VIP Exhibition Room 204. 
[ 4] alice_interview.txt    253 ch | INTERVIEW — ALICE MORGAN Role: Event Coordinator  Alice Morgan stated that she was working
[ 5] timeline.txt           411 ch | CASE TIMELINE — AD-204  21:35 — Aurora Diamond confirmed present by security officer Elena
[ 6] timeline.txt           217 ch | Important observation: Bob Carter stated that he left the hotel at approximately 21:30 and
[ 7] bob_interview.txt      354 ch | INTERVIEW — BOB CARTER Role: Hotel Security Officer  Bob Carter stated: "I finished m

👀 **Look closely:** the security policy's title ends up **alone** in its own chunk. A chunk with no evidence can still be retrieved and waste a slot. Chunking is an engineering choice with side effects.

## 4. Embeddings

An **embedding model** converts text into a list of numbers (a vector). Texts with **similar meaning** get vectors that point in **similar directions**, even when they use different words.

`all-MiniLM-L6-v2` produces **384 numbers** per text. It runs locally on the CPU.

⚠️ The embedding model and the LLM are **two different models** with two different jobs:
- embedding model → turns text into vectors (used for searching)
- LLM → reads the retrieved text and writes the answer

In [9]:
_embedding_cache = {}

def get_embedding(model_name):
    # Load each embedding model only once
    if model_name not in _embedding_cache:
        _embedding_cache[model_name] = HuggingFaceEmbeddings(
            model_name=model_name,
            model_kwargs={"device": "cpu"},
            encode_kwargs={"normalize_embeddings": True},
        )
    return _embedding_cache[model_name]

embedding_model = get_embedding(EMBEDDING_MODEL)

vector = embedding_model.embed_query("Who entered Room 204?")
print("Embedding dimensions:", len(vector))
print("First 10 values:", [round(v, 4) for v in vector[:10]])

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding dimensions: 384
First 10 values: [0.0679, 0.0436, -0.0174, 0.0118, -0.0344, 0.0504, 0.0121, -0.0502, -0.0008, 0.0148]


No single number means anything on its own. What matters is the **direction** of the whole vector compared with other vectors.

## 5. Create the vector database

We store each chunk together with its vector in **Chroma**.

- **Embedding model** → converts text into vectors.
- **Vector database** → stores vectors and finds the nearest ones.
- **LLM** → generates the final answer.

The vector database does not reason about the case. It only measures distances.

**The fix:** `build_vectorstore()` deletes any old collection with the same name before building. We also tell Chroma to use **cosine distance**, which makes the scores easier to read (section 7).

In [10]:
chroma_client = chromadb.EphemeralClient()   # in-memory database for this session

def build_vectorstore(chunks, embedding, name="aurora_diamond_case"):
    # FIX 1: start from an empty collection so chunks are never stored twice
    try:
        chroma_client.delete_collection(name)
    except Exception:
        pass   # the collection did not exist yet
    return Chroma.from_documents(
        documents=chunks,
        embedding=embedding,
        collection_name=name,
        client=chroma_client,
        collection_metadata={"hnsw:space": "cosine"},
    )

vectorstore = build_vectorstore(chunks, embedding_model)
print("Vectors stored:", vectorstore._collection.count(), "| chunks:", len(chunks))

Vectors stored: 11 | chunks: 11


🔁 **Try it:** run the cell above two or three times. The number of stored vectors stays equal to the number of chunks. In the old version it grew every time.

## 6. Inspect similarity search BEFORE using the LLM

This is the most important debugging habit in RAG. First ask:

> **Did retrieval find the right evidence?**

Only after that ask:

> **Did the LLM use the evidence correctly?**

If retrieval is wrong, changing the prompt or the LLM cannot fix the real problem.

In [11]:
question = "Who entered Room 204 at 21:37?"
results = vectorstore.similarity_search(question, k=3)

for i, doc in enumerate(results, start=1):
    print(f"===== RESULT {i}: {name(doc)} =====")
    print(doc.page_content, "\n")

===== RESULT 1: access_logs.txt =====
SECURITY ACCESS LOG — ROOM 204

21:37:12 — Badge B-17 — Bob Carter — ENTRY — Room 204
21:45:03 — Badge B-17 — Bob Carter — EXIT — Room 204

Security system notes:
Badge B-17 is assigned to Bob Carter.
No other badge was recorded entering Room 204 between 21:30 and 21:50. 

===== RESULT 2: david_interview.txt =====
INTERVIEW — DAVID ROSS
Role: Guest

David Ross stated that he was in the main hotel lobby between approximately 21:30 and 21:50.
Lobby camera footage places David in the lobby during the relevant period.

David did not have an authorized security badge for Room 204. 

===== RESULT 3: cctv_report.txt =====
CCTV REVIEW — ROOM 204 CORRIDOR

21:36:58 — A person wearing a black hotel security uniform approaches Room 204.
21:37:14 — The person enters Room 204.
21:41:50 — The person remains inside the room area.
21:44:47 — The person leaves Room 204.
21:45:05 — The corridor is empty.

The camera angle does not provide a sufficiently clear facial

✅ Three **different** chunks now come back. In the old version you got the access log three times because of the duplicates.

## 7. Read the similarity scores

Because we chose **cosine** distance, the numbers mean:

| Value | Meaning |
|---|---|
| **distance** (`similarity_search_with_score`) | `0` = same direction, larger = less similar. **Smaller is better.** |
| **relevance** (`similarity_search_with_relevance_scores`) | `1 − distance`. **Larger is better.** |

⚠️ Neither is a confidence percentage. A relevance of 0.45 does **not** mean "45 % sure". Compare scores **with each other** for the same question.

In [12]:
pairs = vectorstore.similarity_search_with_relevance_scores(
    "What evidence contradicts Bob Carter's statement?", k=5
)
for doc, score in pairs:
    print(f"relevance {score:.3f} | distance {1 - score:.3f} | {name(doc)}")

relevance 0.357 | distance 0.643 | bob_interview.txt
relevance 0.237 | distance 0.763 | timeline.txt
relevance 0.196 | distance 0.804 | timeline.txt
relevance 0.160 | distance 0.840 | access_logs.txt
relevance 0.066 | distance 0.934 | david_interview.txt


## 8. The retriever

A **retriever** is the component that fetches documents for a question. We write one function that supports three **search types**; Part B explains each in detail.

| `search_type` | How it chooses chunks |
|---|---|
| `"similarity"` | The `k` closest chunks. |
| `"mmr"` | Relevant chunks that are also **different from each other**. |
| `"threshold"` | Every chunk whose relevance is above `score_threshold` (up to `k`). |

In [13]:
def retrieve(question, vs=None, k=TOP_K, search_type="similarity",
             fetch_k=20, lambda_mult=0.5, score_threshold=0.3):
    vs = vectorstore if vs is None else vs
    if search_type == "similarity":
        return vs.similarity_search(question, k=k)
    if search_type == "mmr":
        return vs.max_marginal_relevance_search(
            question, k=k, fetch_k=fetch_k, lambda_mult=lambda_mult)
    if search_type == "threshold":
        pairs = vs.similarity_search_with_relevance_scores(question, k=k)
        return [doc for doc, score in pairs if score >= score_threshold]
    raise ValueError("search_type must be 'similarity', 'mmr' or 'threshold'")

found = retrieve("What evidence contradicts Bob Carter's statement?")
print("Retrieved:", [name(d) for d in found])

Retrieved: ['bob_interview.txt', 'timeline.txt', 'timeline.txt', 'access_logs.txt', 'david_interview.txt']


## 9. Connect Groq (the LLM)

Now the **generation** layer. The LLM receives the question, the retrieved evidence, and instructions for using the evidence.

- `temperature` — 0 means the model always picks the most likely next word, so answers are repeatable.
- `max_tokens` — the maximum length of the answer. A token is roughly ¾ of an English word.

`ask()` retries automatically if Groq's free tier says we are sending requests too fast.

In [14]:
def make_llm(model=LLM_MODEL, temperature=TEMPERATURE, max_tokens=MAX_TOKENS):
    return ChatGroq(model=model, temperature=temperature, max_tokens=max_tokens)

def ask(llm, prompt, retries=3):
    for attempt in range(retries):
        try:
            return llm.invoke(prompt)
        except Exception as e:
            if "rate" in str(e).lower() and attempt < retries - 1:
                print("Rate limit reached — waiting 20 seconds…")
                time.sleep(20)
            else:
                raise

llm = make_llm()
print("Groq LLM ready:", LLM_MODEL)

Groq LLM ready: openai/gpt-oss-120b


## 10. The RAG prompt

The prompt separates **evidence**, **reasoning** and **conclusion**, and it sets rules. This is **grounding**: the answer must come from the retrieved evidence, not from the model's general knowledge.

Each rule has a number so that in Part B we can switch rules **off** one at a time and see what each one does.

In [15]:
RULES = {
    1: "Do not invent facts.",
    2: "Distinguish documented facts from conclusions.",
    3: "If the evidence is insufficient, explicitly say so.",
    4: "Identify contradictions between statements and records.",
    5: "Important factual claims should include the source filename.",
    6: "Do not state that a person is definitely guilty unless the evidence proves that.",
    7: "You may identify the most likely suspect when the evidence supports that conclusion.",
    8: "Explain the reasoning clearly.",
}

def build_prompt(question, retrieved_docs, rules=None):
    rules = RULES if rules is None else rules
    context = "\n\n".join(
        f"[SOURCE {i}: {name(doc)}]\n{doc.page_content}"
        for i, doc in enumerate(retrieved_docs, start=1)
    )
    rule_text = "\n".join(f"{i}. {text}" for i, text in enumerate(rules.values(), start=1))
    return f'''You are an evidence-based investigation assistant.

Answer the question using ONLY the evidence provided below.

Rules:
{rule_text}

QUESTION:
{question}

EVIDENCE:
{context}

Return the answer using this structure:

CONCLUSION:
...

EVIDENCE:
- ...

CONTRADICTIONS:
- ...

SOURCES:
- ...
'''

print(build_prompt("Who entered Room 204?", retrieve("Who entered Room 204?", k=2)))

You are an evidence-based investigation assistant.

Answer the question using ONLY the evidence provided below.

Rules:
1. Do not invent facts.
2. Distinguish documented facts from conclusions.
3. If the evidence is insufficient, explicitly say so.
4. Identify contradictions between statements and records.
5. Important factual claims should include the source filename.
6. Do not state that a person is definitely guilty unless the evidence proves that.
7. You may identify the most likely suspect when the evidence supports that conclusion.
8. Explain the reasoning clearly.

QUESTION:
Who entered Room 204?

EVIDENCE:
[SOURCE 1: david_interview.txt]
INTERVIEW — DAVID ROSS
Role: Guest

David Ross stated that he was in the main hotel lobby between approximately 21:30 and 21:50.
Lobby camera footage places David in the lobby during the relevant period.

David did not have an authorized security badge for Room 204.

[SOURCE 2: alice_interview.txt]
INTERVIEW — ALICE MORGAN
Role: Event Coordinat

## 11. The complete RAG function

**Question → Retrieval → Context → Groq → Grounded answer**

`rag()` returns a dictionary, so we can inspect every part of the result, not only the answer text.
If retrieval returns **nothing** (possible with a score threshold), we skip the LLM call and say so. Why pay for an answer with no evidence?

In [16]:
def rag(question, vs=None, llm_=None, rules=None, **retrieval):
    found = retrieve(question, vs=vs, **retrieval)
    if not found:
        return {"answer": "CONCLUSION:\nNo evidence passed the retrieval filter, so the question cannot be answered.",
                "docs": [], "sources": [], "finish_reason": "skipped: no evidence"}
    prompt = build_prompt(question, found, rules=rules)
    response = ask(llm_ or llm, prompt)
    return {
        "answer": response.content,
        "docs": found,
        "sources": [name(d) for d in found],
        "finish_reason": response.response_metadata.get("finish_reason"),
        "prompt": prompt,
    }

def section(answer, title):
    # Pull one section (e.g. 'CONTRADICTIONS') out of a structured answer
    m = re.search(rf"\**{title}:?\**\s*(.*?)(?=\n\s*\**(?:CONCLUSION|EVIDENCE|CONTRADICTIONS|SOURCES)\b|\Z)",
                  answer, flags=re.S | re.I)
    return m.group(1).strip() if m else "(section not found)"

## 12. Test the investigation

Before reading each answer, look at **which sources were retrieved**. Then judge the answer.

In [17]:
Q1 = ("Who is the most likely person responsible for the disappearance of the Aurora Diamond? "
      "Provide the strongest evidence and identify any statement that is contradicted by the records.")

result = rag(Q1)
print("RETRIEVED:", result["sources"], "\n")
print(result["answer"])

RETRIEVED: ['incident_report.txt', 'timeline.txt', 'bob_interview.txt', 'alice_interview.txt', 'timeline.txt'] 

**CONCLUSION:**  
The person most likely responsible for the disappearance of the Aurora Diamond is **Bob Carter (or an individual using his security badge B‑17)**. The badge was recorded entering and exiting the display room at the exact times the diamond went missing, and a person in a black security uniform (consistent with a hotel security officer) was captured on CCTV inside the room during that interval. Bob’s own statement that he left the hotel at 21:30 and never returned directly contradicts the electronic access logs, making his account unreliable.

**EVIDENCE:**  
- The diamond was confirmed present at 21:35 and missing at 21:42 (incident_report.txt).  
- Badge B‑17 (assigned to Bob Carter) recorded **entry at 21:37** and **exit at 21:45** from Room 204 (timeline.txt).  
- CCTV captured a **person in a black hotel security uniform** inside Room 204 at 21:38 and le

🤔 **Check:** is `bob_interview.txt` among the retrieved sources? If not, the model cannot know that Bob *claimed* he left at 21:30, so it cannot report that contradiction. Keep this in mind: it is the main thread of Part B.

### Question 2 — Find a contradiction

In [18]:
result = rag("What evidence contradicts Bob Carter's statement that he left the hotel at approximately 21:30?")
print("RETRIEVED:", result["sources"], "\n")
print(result["answer"])

RETRIEVED: ['bob_interview.txt', 'timeline.txt', 'timeline.txt', 'access_logs.txt', 'david_interview.txt'] 

**CONCLUSION:**  
The electronic access records and CCTV footage show that Badge B‑17 (assigned to Bob Carter) was used to enter and exit Room 204 after 21:30, directly contradicting Bob Carter’s statement that he left the hotel at approximately 21:30 and did not return to the room.

**EVIDENCE:**  
- **Badge entry:** 21:37:12 – Badge B‑17 recorded entry into Room 204. (Source 4: access_logs.txt)  
- **Badge exit:** 21:45:03 – Badge B‑17 recorded exit from Room 204. (Source 4: access_logs.txt)  
- **CCTV observation:** 21:38 – CCTV shows a person in a black hotel security uniform inside Room 204. (Source 3: timeline.txt)  
- **CCTV observation:** 21:44 – CCTV shows the same person leaving Room 204. (Source 3: timeline.txt)  
- **Access log note:** No other badge was recorded entering Room 204 between 21:30 and 21:50, confirming that the only badge used was Bob Carter’s. (Source 

### Question 3 — Ask for information that does not exist
A good grounded system must **refuse to invent** an answer.

In [19]:
CAR_Q = "What color was Bob Carter's car on the night of the incident?"
result = rag(CAR_Q)
print("RETRIEVED:", result["sources"], "\n")
print(result["answer"])

RETRIEVED: ['bob_interview.txt', 'timeline.txt', 'access_logs.txt', 'timeline.txt', 'cctv_report.txt'] 

**CONCLUSION:**  
The evidence provided does not contain any information about the color of Bob Carter’s car on the night of the incident. Therefore, the car’s color cannot be determined from the available records.

**EVIDENCE:**  
- The interview transcript (bob_interview.txt) discusses Bob Carter’s badge and movements but makes no mention of a vehicle.  
- The timeline (timeline.txt) and access logs (access_logs.txt) record badge entries, CCTV observations, and the theft of the Aurora Diamond, but contain no reference to a car or its color.  
- The CCTV report (cctv_report.txt) describes a person in a black security uniform in the corridor, with no details about any vehicle.  

**CONTRADICTIONS:**  
- None identified regarding the color of a car, as no such information is present in any source.

**SOURCES:**  
- bob_interview.txt  
- timeline.txt (both sections)  
- access_logs.tx

### Question 4 — Test another suspect

In [20]:
result = rag("Could Clara Lee have entered Room 204 according to the available evidence?")
print("RETRIEVED:", result["sources"], "\n")
print(result["answer"])

RETRIEVED: ['clara_interview.txt', 'alice_interview.txt', 'timeline.txt', 'timeline.txt', 'cctv_report.txt'] 

**CONCLUSION:**  
The available evidence does not support the possibility that Clara Lee entered Room 204. There is no record of her having a badge for that room, no visual identification of her in the CCTV footage, and her documented activity places her photographing near Room 203 at times that overlap the period when a security‑uniformed individual (identified by badge B‑17) was recorded entering and exiting Room 204.

**EVIDENCE:**  
- *clara_interview.txt* – Clara says she was photographing guests **near Room 203**; her camera timestamps are 21:36, 21:39, 21:43. She **did not have an authorized security badge for Room 204**.  
- *timeline.txt* – Badge B‑17 (assigned to Bob Carter) records entry into Room 204 at 21:37 and exit at 21:45.  
- *cctv_report.txt* – CCTV shows a person in a **black hotel security uniform** entering Room 204 at 21:37:14, remaining inside, and leav

## 13. Evaluate retrieval

We measure retrieval **separately** from the answer. For each test question we know which file(s) contain the answer. **Recall@k** is the share of questions where at least one correct file appears in the top `k`.

The first version printed 0 % because it compared `mystery_data/access_logs.txt` with `access_logs.txt`. Now we compare file names, and some questions accept two files (the timeline repeats several facts).

We also report **rank**: the position of the first correct file (1 is best; `–` means not found).

In [21]:
EVAL_SET = [
    ("Who entered Room 204 at 21:37?",                         {"access_logs.txt", "timeline.txt"}),
    ("When was the diamond confirmed to be present?",          {"incident_report.txt", "timeline.txt"}),
    ("What contradicts Bob's statement that he left at 21:30?", {"access_logs.txt", "timeline.txt"}),
    ("Who was photographing guests near Room 203?",            {"clara_interview.txt"}),
    ("What are the rules for entering Room 204 after 21:30?",  {"security_policy.txt"}),
    ("Where was David Ross during the incident?",              {"david_interview.txt"}),
    ("Could the CCTV camera identify the person's face?",      {"cctv_report.txt"}),
    ("What did Bob say about his security badge?",             {"bob_interview.txt"}),
    ("Where was Alice Morgan working?",                        {"alice_interview.txt"}),
    ("Was there any sign of forced entry?",                    {"incident_report.txt"}),
]

def evaluate(vs=None, k=3, show=False, **retrieval):
    rows = []
    for q, expected in EVAL_SET:
        got = [name(d) for d in retrieve(q, vs=vs, k=k, **retrieval)]
        rank = next((i for i, f in enumerate(got, start=1) if f in expected), None)
        rows.append({"question": q, "expected": ", ".join(sorted(expected)),
                     "retrieved": ", ".join(got), "hit": rank is not None, "rank": rank or "–"})
    table = pd.DataFrame(rows)
    if show:
        display(table)
    return table["hit"].mean()

print(f"Recall@3 = {evaluate(k=3, show=True):.0%}")

,question,expected,retrieved,hit,rank
0,Who entered Room 204 at 21:37?,"access_logs.txt, timeline.txt","access_logs.txt, david_interview.txt, cctv_report.txt",True,1
1,When was the diamond confirmed to be present?,"incident_report.txt, timeline.txt","incident_report.txt, timeline.txt, bob_interview.txt",True,1
2,What contradicts Bob's statement that he left at 21:30?,"access_logs.txt, timeline.txt","bob_interview.txt, timeline.txt, access_logs.txt",True,2
3,Who was photographing guests near Room 203?,clara_interview.txt,"clara_interview.txt, cctv_report.txt, david_interview.txt",True,1
4,What are the rules for entering Room 204 after 21:30?,security_policy.txt,"cctv_report.txt, access_logs.txt, david_interview.txt",False,–
5,Where was David Ross during the incident?,david_interview.txt,"david_interview.txt, bob_interview.txt, timeline.txt",True,1
6,Could the CCTV camera identify the person's face?,cctv_report.txt,"cctv_report.txt, clara_interview.txt, timeline.txt",True,1
7,What did Bob say about his security badge?,bob_interview.txt,"bob_interview.txt, access_logs.txt, timeline.txt",True,1
8,Where was Alice Morgan working?,alice_interview.txt,"alice_interview.txt, timeline.txt, bob_interview.txt",True,1
9,Was there any sign of forced entry?,incident_report.txt,"timeline.txt, access_logs.txt, bob_interview.txt",False,–


Recall@3 = 80%


✍️ **Look at any misses.** Was the right file at rank 4 or 5? Was the question worded differently from the document? These observations guide the tuning in Part B.

## 🧠 RAG architecture summary

```text
                 USER QUESTION
                       │
                       ▼
            ┌─────────────────────┐
            │  Embedding model    │  ← EMBEDDING_MODEL
            └──────────┬──────────┘
                       ▼
            ┌─────────────────────┐
            │  Chroma vector DB   │  ← CHUNK_SIZE, CHUNK_OVERLAP
            └──────────┬──────────┘
                       │  top-k chunks   ← TOP_K, search_type
                       ▼
            ┌─────────────────────┐
            │  Prompt + context   │  ← RULES
            └──────────┬──────────┘
                       ▼
            ┌─────────────────────┐
            │  Groq LLM           │  ← LLM_MODEL, TEMPERATURE, MAX_TOKENS
            └──────────┬──────────┘
                       ▼
          GROUNDED ANSWER + SOURCES
```

**RAG is an architecture, not a model.** It combines documents, a chunking strategy, an embedding model, a vector store, a retrieval strategy, a prompt, an LLM, and evaluation. Every arrow above has a setting you can tune.

---
# Part B — Tuning lab 🔬

## Tuning is not fine-tuning
- **Tuning** (this lab): changing the *settings* of the pipeline. Fast, free, no training. You can try a new value in seconds.
- **Fine-tuning**: *retraining* a model's internal weights on new examples. Slow, needs a GPU and training data, and changes the model itself.

In RAG, you almost always tune first. Most problems are solved by better chunks, better retrieval or a better prompt.

## Rules of the lab
1. **Change one setting at a time.** If you change two, you cannot tell which one caused the effect.
2. **Predict before you run.** Write your guess in the ✍️ line. Being wrong is where learning happens.
3. **Measure, don't eyeball.** Every experiment produces a table.
4. **Keep the question fixed.** Most experiments use Q1, the "who did it" question.

### Our main measurement: the Q1 evidence checklist
A complete answer to Q1 needs these five facts. We check which ones reach the LLM. If a fact is not in the retrieved chunks, the LLM cannot use it, however smart it is.

In [22]:
Q1_FACTS = {
    "Badge B-17 entered/exited":        r"B-17",
    "Uniformed person on CCTV":         r"security uniform",
    "Bob's claim: left at 21:30":       r"left the hotel",
    "CCTV cannot identify the face":    r"facial image",
    "Policy: badge holder responsible": r"responsible for access",
}

def facts_found(found_docs, facts=Q1_FACTS):
    text = "\n".join(d.page_content for d in found_docs)
    return [fact for fact, pattern in facts.items() if re.search(pattern, text)]

def unique_files(found_docs):
    return list(dict.fromkeys(name(d) for d in found_docs))

lab_results = {}   # every experiment saves its table here

baseline_docs = retrieve(Q1)
print("Baseline Q1 sources:", [name(d) for d in baseline_docs])
print(f"Facts reaching the LLM: {len(facts_found(baseline_docs))}/5 →", facts_found(baseline_docs))

Baseline Q1 sources: ['incident_report.txt', 'timeline.txt', 'bob_interview.txt', 'alice_interview.txt', 'timeline.txt']
Facts reaching the LLM: 3/5 → ['Badge B-17 entered/exited', 'Uniformed person on CCTV', "Bob's claim: left at 21:30"]


## B1. `chunk_size` — how big is each piece?

**What it means:** the maximum number of characters in one chunk.

| Small chunks | Large chunks |
|---|---|
| ✅ precise: a chunk is about one thing | ✅ context stays together |
| ❌ a sentence can lose its context (who said it?) | ❌ one chunk mixes relevant and irrelevant text |
| ❌ you need a larger `k` to collect all facts | ❌ fewer, longer chunks fill the prompt faster |

✍️ **Predict:** at `chunk_size=1000`, how many chunks will there be? At 100?

In [23]:
rows = []
for size in [100, 150, 200, 300, 500, 1000]:
    overlap = min(CHUNK_OVERLAP, size // 5)          # overlap must stay smaller than the chunk
    exp_chunks = make_chunks(size, overlap)
    vs_exp = build_vectorstore(exp_chunks, embedding_model, name="exp_chunks")
    found = retrieve(Q1, vs=vs_exp, k=TOP_K)
    rows.append({
        "chunk_size": size,
        "chunks": len(exp_chunks),
        "avg length": round(sum(len(c.page_content) for c in exp_chunks) / len(exp_chunks)),
        "recall@3": f"{evaluate(vs_exp, k=3):.0%}",
        f"Q1 facts in top {TOP_K}": f"{len(facts_found(found))}/5",
        "Q1 files": ", ".join(f.replace('.txt', '') for f in unique_files(found)),
    })

lab_results["B1 chunk_size"] = pd.DataFrame(rows)
display(lab_results["B1 chunk_size"])

,chunk_size,chunks,avg length,recall@3,Q1 facts in top 5,Q1 files
0,100,51,68,90%,0/5,"incident_report, timeline, bob_interview"
1,150,41,84,90%,2/5,"incident_report, bob_interview, timeline"
2,200,32,108,90%,2/5,"incident_report, bob_interview, timeline"
3,300,19,187,90%,2/5,"incident_report, timeline, bob_interview"
4,500,11,316,80%,3/5,"incident_report, timeline, bob_interview, alice_interview"
5,1000,9,387,80%,3/5,"incident_report, timeline, bob_interview, alice_interview, clara_interview"


🔍 **Zoom in:** here is Bob's interview cut at `chunk_size=150`. Read each chunk on its own, the way the LLM would see it if only that chunk were retrieved.

In [24]:
for c in make_chunks(150, 30):
    if name(c) == "bob_interview.txt":
        print("─" * 60)
        print(c.page_content)

────────────────────────────────────────────────────────────
INTERVIEW — BOB CARTER
Role: Hotel Security Officer
────────────────────────────────────────────────────────────
Bob Carter stated:
"I finished my work around 21:30 and left the hotel.
I did not return to Room 204 after that.
────────────────────────────────────────────────────────────
My security badge stayed with me the entire evening.
Nobody else had access to my badge."
────────────────────────────────────────────────────────────
Bob said he was not aware that the Aurora Diamond was missing until another officer informed him.


**What to look for**
- At **1000**, every file is one chunk. Retrieval becomes "pick a whole file".
- At **100–150**, a chunk like *"I did not return to Room 204 after that."* no longer says **who** said it.
- More chunks means that with a fixed `k`, each answer sees a smaller share of the case.

**Questions**
1. Which size gave the most Q1 facts with `k = 5`? Is that also the best recall@3?
2. Would the best size be the same for a 300-page hotel manual? Why not?

## B2. `chunk_overlap` — insurance at the cut

**What it means:** how many characters at the end of one chunk are repeated at the start of the next. If a key sentence is cut in half, the overlap gives the next chunk a second chance to contain it whole.

The cost: the same text is stored (and maybe retrieved) twice.

We test at `chunk_size=150`, where cuts happen often.

✍️ **Predict:** will overlap improve recall on our small case files?

In [25]:
original_chars = sum(len(d.page_content) for d in docs)
rows = []
for overlap in [0, 30, 75]:
    exp_chunks = make_chunks(150, overlap)
    vs_exp = build_vectorstore(exp_chunks, embedding_model, name="exp_overlap")
    stored = sum(len(c.page_content) for c in exp_chunks)
    rows.append({
        "chunk_overlap": overlap,
        "chunks": len(exp_chunks),
        "text stored vs original": f"{stored / original_chars:.0%}",
        "recall@3": f"{evaluate(vs_exp, k=3):.0%}",
        f"Q1 facts in top {TOP_K}": f"{len(facts_found(retrieve(Q1, vs=vs_exp)))}/5",
    })

lab_results["B2 chunk_overlap"] = pd.DataFrame(rows)
display(lab_results["B2 chunk_overlap"])

,chunk_overlap,chunks,text stored vs original,recall@3,Q1 facts in top 5
0,0,41,99%,90%,2/5
1,30,41,99%,90%,2/5
2,75,43,108%,80%,2/5


**What to look for:** our documents are short and line-based, and the splitter already cuts at line breaks. Overlap may change **very little**. That is a valid result.

**Lesson:** a setting can be important in general and irrelevant for *your* data. You only know by measuring.

**Question:** what kind of documents would make overlap really matter? (Hint: long paragraphs with no line breaks.)

## B3. `k` — how many chunks reach the LLM?

**What it means:** the number of chunks the retriever passes to the LLM. Everything outside the top `k` is invisible to the model.

It is the same problem as the evidence board in the homework game: limited slots.

| Small `k` | Large `k` |
|---|---|
| ✅ short, focused prompt | ✅ more facts reach the LLM |
| ❌ key evidence can be cut off | ❌ more noise, longer and costlier prompts |

✍️ **Predict:** what is the smallest `k` at which Bob's claim ("left the hotel") reaches the LLM?

In [26]:
rows = []
for k in [1, 3, 5, 7, 9]:
    found = retrieve(Q1, k=k)
    facts = facts_found(found)
    rows.append({
        "k": k,
        "different files": len(unique_files(found)),
        "facts": f"{len(facts)}/5",
        "Bob's claim included": "Bob's claim: left at 21:30" in facts,
        "files": ", ".join(f.replace('.txt', '') for f in unique_files(found)),
    })

lab_results["B3 k"] = pd.DataFrame(rows)
display(lab_results["B3 k"])

,k,different files,facts,Bob's claim included,files
0,1,1,0/5,False,incident_report
1,3,3,3/5,True,"incident_report, timeline, bob_interview"
2,5,4,3/5,True,"incident_report, timeline, bob_interview, alice_interview"
3,7,6,4/5,True,"incident_report, timeline, bob_interview, alice_interview, clara_interview, security_policy"
4,9,8,4/5,True,"incident_report, timeline, bob_interview, alice_interview, clara_interview, security_policy, access_logs, david_inte..."


Now compare the **CONTRADICTIONS** section of the answer at a small and a large `k`.
Set `BEST_K` to the smallest `k` that included Bob's claim in the table above.

In [27]:
BEST_K = 7   # ✍️ change this to what you found above

for k in [3, BEST_K]:
    out = rag(Q1, k=k)
    print(f"===== k = {k} | sources: {unique_files(out['docs'])}")
    print("CONTRADICTIONS:", section(out["answer"], "CONTRADICTIONS"), "\n")

===== k = 3 | sources: ['incident_report.txt', 'timeline.txt', 'bob_interview.txt']
CONTRADICTIONS: - Bob Carter’s interview states: “I finished my work around **21:30** and left the hotel. I did not return to Room 204 after that. My security badge stayed with me the entire evening.” *([bob_interview.txt])*  
- This statement directly conflicts with the **badge logs** (entry at 21:37, exit at 21:45) and the **CCTV evidence** (person in security uniform inside the room at 21:38 and leaving at 21:44). 

===== k = 7 | sources: ['incident_report.txt', 'timeline.txt', 'bob_interview.txt', 'alice_interview.txt', 'clara_interview.txt', 'security_policy.txt']
CONTRADICTIONS: - **Bob Carter’s statement** that he left the hotel at ~21:30, never returned to Room 204, and kept his badge with him **contradicts** the electronic access logs showing his badge was used to enter and exit the room (SOURCE 5, 2).  
- The claim that “nobody else had access to my badge” is also contradicted by the badge’s r

**Key insight:** when the model says *"no contradictions"*, it may be telling the truth about **what it was shown**. The bug is in retrieval, not in the LLM.

**Question:** a bigger `k` fixed this case. Name one cost of always using a large `k` in a real product.

## B4. Search type: similarity vs **MMR**

**Similarity** returns the `k` closest chunks, even if they say nearly the same thing. Our `timeline.txt` repeats facts from the access log and the CCTV report, so similarity can spend slots on the same fact twice.

**MMR (Maximal Marginal Relevance)** works in two steps:
1. fetch the `fetch_k` most similar chunks (a candidate pool);
2. pick `k` of them, one by one, preferring chunks that are relevant **and different from those already picked**.

`lambda_mult` sets the balance:
- `1.0` → only relevance (same as similarity)
- `0.0` → only variety
- `0.5` → a balance

✍️ **Predict:** can MMR bring Bob's interview into the top 5 **without** increasing `k`?

In [28]:
rows = []
settings = [("similarity", None)] + [("mmr", lam) for lam in [1.0, 0.7, 0.5, 0.3, 0.0]]
for search_type, lam in settings:
    kwargs = {"search_type": search_type, "k": TOP_K}
    if lam is not None:
        kwargs["lambda_mult"] = lam
    found = retrieve(Q1, **kwargs)
    rows.append({
        "search": search_type if lam is None else f"mmr λ={lam}",
        "different files": len(unique_files(found)),
        "facts": f"{len(facts_found(found))}/5",
        "recall@3": f"{evaluate(**(kwargs | {'k': 3})):.0%}",
        "files": ", ".join(f.replace('.txt', '') for f in unique_files(found)),
    })

lab_results["B4 MMR"] = pd.DataFrame(rows)
display(lab_results["B4 MMR"])

,search,different files,facts,recall@3,files
0,similarity,4,3/5,80%,"incident_report, timeline, bob_interview, alice_interview"
1,mmr λ=1.0,4,3/5,80%,"incident_report, timeline, bob_interview, alice_interview"
2,mmr λ=0.7,5,3/5,90%,"incident_report, timeline, bob_interview, alice_interview, clara_interview"
3,mmr λ=0.5,5,3/5,90%,"incident_report, timeline, bob_interview, alice_interview, security_policy"
4,mmr λ=0.3,5,1/5,90%,"incident_report, bob_interview, alice_interview, david_interview, security_policy"
5,mmr λ=0.0,5,1/5,80%,"incident_report, alice_interview, clara_interview, access_logs, security_policy"


**What to look for**
- Does the number of *different files* grow as λ goes down?
- At some λ, do Alice's or David's interviews appear? They are "different", but are they **useful** for Q1?
- Does recall@3 suffer? Variety can push the single best chunk out.

**Question:** MMR improves variety. When is variety bad?

## B5. Search type: **score threshold**

Instead of "always take `k` chunks", take **only chunks whose relevance is above a minimum**. The number of retrieved chunks now changes per question, and it can be **zero**.

First, look at the real relevance scores for three questions. A threshold only makes sense once you know the range.

In [29]:
for q in [Q1, "Where was David Ross during the incident?", CAR_Q]:
    pairs = vectorstore.similarity_search_with_relevance_scores(q, k=len(chunks))
    print(f"\n{q[:70]}")
    for doc, score in pairs[:6]:
        print(f"   {score:.3f}  {name(doc)}")


Who is the most likely person responsible for the disappearance of the
   0.589  incident_report.txt
   0.484  timeline.txt
   0.371  bob_interview.txt
   0.206  alice_interview.txt
   0.182  timeline.txt
   0.147  clara_interview.txt

Where was David Ross during the incident?
   0.548  david_interview.txt
   0.310  bob_interview.txt
   0.263  timeline.txt
   0.247  timeline.txt
   0.229  incident_report.txt
   0.220  access_logs.txt

What color was Bob Carter's car on the night of the incident?
   0.437  bob_interview.txt
   0.373  timeline.txt
   0.315  access_logs.txt
   0.285  timeline.txt
   0.247  cctv_report.txt
   0.234  incident_report.txt


✍️ **Predict:** pick a threshold that keeps Q1's useful chunks but returns **nothing** for the car question. Write it here, then test it below.

In [30]:
rows = []
questions = {"Q1 (who did it)": Q1, "David's location": "Where was David Ross during the incident?", "Car colour": CAR_Q}
for threshold in [0.1, 0.2, 0.3, 0.4, 0.5]:
    row = {"threshold": threshold}
    for label, q in questions.items():
        row[label] = len(retrieve(q, search_type="threshold", k=10, score_threshold=threshold))
    rows.append(row)

lab_results["B5 threshold"] = pd.DataFrame(rows)
print("Number of chunks retrieved:")
display(lab_results["B5 threshold"])

Number of chunks retrieved:


,threshold,Q1 (who did it),David's location,Car colour
0,0.1,8,10,9
1,0.2,4,6,7
2,0.3,3,2,3
3,0.4,2,1,1
4,0.5,1,1,0


In [31]:
THRESHOLD = 0.3   # ✍️ set the value you chose

out = rag(CAR_Q, search_type="threshold", k=10, score_threshold=THRESHOLD)
print("Retrieved:", out["sources"], "| finish:", out["finish_reason"])
print(out["answer"])

Retrieved: ['bob_interview.txt', 'timeline.txt', 'access_logs.txt'] | finish: stop
**CONCLUSION:**  
The evidence provided does not contain any information about the color of Bob Carter’s car on the night of the incident. Therefore, the car’s color cannot be determined from the available records.

**EVIDENCE:**  
- *bob_interview.txt* – contains only Bob Carter’s statements about his work schedule, badge possession, and awareness of the missing diamond; no mention of a car or its color.  
- *timeline.txt* – details the timeline of the Aurora Diamond theft and badge activity; no reference to a vehicle.  
- *access_logs.txt* – records badge entries and exits for Room 204; no vehicle information is included.

**CONTRADICTIONS:**  
- None. The sources are consistent with each other regarding badge usage and the timeline, but none address the car’s color.

**SOURCES:**  
- bob_interview.txt  
- timeline.txt  
- access_logs.txt


**What to look for:** a threshold is a **second line of defence against hallucination**. If nothing relevant exists, nothing is retrieved and the LLM is never asked. Prompt rule 3 is the first line.

**Questions**
1. Does one threshold work well for **all** questions? Usually not. Why?
2. What happens if you set it too high for an important question?

## B6. The embedding model — what counts as "similar"?

The embedding model decides which chunks are close to the question. Change it and retrieval changes, even with identical chunks and `k`.

| Model | Dimensions | Note |
|---|---|---|
| `all-MiniLM-L6-v2` | 384 | small and fast; our baseline; English |
| `all-mpnet-base-v2` | 768 | larger and usually stronger; English |
| `BAAI/bge-small-en-v1.5` | 384 | newer, trained for retrieval; English |
| `paraphrase-multilingual-MiniLM-L12-v2` | 384 | understands 50+ languages, including Russian |

⚠️ When you change the embedding model you must **re-embed every chunk**. Vectors from different models live in different "spaces" and cannot be compared.

We also test a question in **Russian**, while all the documents are in English.

✍️ **Predict:** which model will find `access_logs.txt` for the Russian question?

⏳ The first run downloads each model (up to about 500 MB). It takes a few minutes.

In [32]:
RU_Q = "Кто вошёл в комнату 204 в 21:37?"   # "Who entered Room 204 at 21:37?"

MODELS = [
    "sentence-transformers/all-MiniLM-L6-v2",
    "sentence-transformers/all-mpnet-base-v2",
    "BAAI/bge-small-en-v1.5",
    "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
]

rows = []
for model_name in MODELS:
    start = time.time()
    emb = get_embedding(model_name)
    dims = len(emb.embed_query("test"))
    vs_exp = build_vectorstore(chunks, emb, name="exp_embedding")
    seconds = time.time() - start
    rows.append({
        "model": model_name.split("/")[-1],
        "dims": dims,
        "load + index (s)": round(seconds, 1),
        "recall@3": f"{evaluate(vs_exp, k=3):.0%}",
        f"Q1 facts in top {TOP_K}": f"{len(facts_found(retrieve(Q1, vs=vs_exp)))}/5",
        "Russian Q → top 3": ", ".join(f.replace('.txt', '') for f in unique_files(retrieve(RU_Q, vs=vs_exp, k=3))),
    })

lab_results["B6 embedding"] = pd.DataFrame(rows)
display(lab_results["B6 embedding"])

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/11.6k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

...

tokenizer.json: reconstructing file:   0%|          |  0.00B / 9.08MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

,model,dims,load + index (s),recall@3,Q1 facts in top 5,Russian Q → top 3
0,all-MiniLM-L6-v2,384,0.4,80%,3/5,"clara_interview, access_logs, david_interview"
1,all-mpnet-base-v2,768,10.0,90%,3/5,"access_logs, david_interview, timeline"
2,bge-small-en-v1.5,384,4.6,90%,3/5,"cctv_report, timeline, access_logs"
3,paraphrase-multilingual-MiniLM-L12-v2,384,8.9,90%,3/5,"timeline, clara_interview, cctv_report"


**What to look for**
- Is the 768-dimension model clearly better on nine short files? Bigger is not automatically better.
- Which models understand the Russian question? English-only models often fail completely on other languages.
- Speed: in a real product with millions of chunks, embedding time and vector size cost money.

**Questions**
1. Your university helpdesk receives questions in Kyrgyz, Russian and English. Which kind of embedding model do you need?
2. Try your own question in Kyrgyz with the multilingual model. Does it work?

## B7. `temperature` — how random is the answer?

**What it means:** when the LLM picks the next word, temperature controls how adventurous it is.
- `0` → always the most likely word. Same prompt, (almost) the same answer.
- `1.0` → more varied wording, sometimes surprising choices.

Temperature does **not** add knowledge. It only adds variation.

We keep retrieval fixed (`k = BEST_K`) and run Q1 three times per temperature.

✍️ **Predict:** at temperature 1.0, will the suspect ever change?

In [33]:
rows = []
for temp in [0, 1.0]:
    llm_t = make_llm(temperature=temp)
    conclusions = []
    for run in range(3):
        out = rag(Q1, llm_=llm_t, k=BEST_K)
        conclusions.append(section(out["answer"], "CONCLUSION"))
    rows.append({
        "temperature": temp,
        "different answers (of 3)": len(set(conclusions)),
        "named Bob (of 3)": sum("Bob" in c for c in conclusions),
        "first conclusion": conclusions[0][:200],
    })

lab_results["B7 temperature"] = pd.DataFrame(rows)
display(lab_results["B7 temperature"])

,temperature,different answers (of 3),named Bob (of 3),first conclusion
0,0.0,3,3,"The person most likely responsible for the disappearance of the Aurora Diamond is **Bob Carter**, the hotel security..."
1,1.0,3,3,Bob Carter is the most likely person responsible for the disappearance of the Aurora Diamond.


**What to look for:** even at 0, answers can differ slightly between runs because of how servers batch requests. At 1.0 the wording changes. Check whether the **facts** stay the same.

**Question:** for an evidence-based assistant, which temperature would you choose? When might a higher temperature be useful? (Think: brainstorming, creative writing.)

## B8. `max_tokens` — how long may the answer be?

**What it means:** a hard limit on how many tokens the LLM may generate. When the limit is reached, the answer simply **stops**, even mid-sentence.

`finish_reason` tells you why the answer ended:
- `stop` → the model finished on its own
- `length` → it hit `max_tokens` and was cut off

⚠️ `gpt-oss-120b` is a **reasoning model**: it may "think" before writing, and that hidden thinking can also use up tokens. With a very small limit you may get a strangely short, or even empty, answer.

✍️ **Predict:** at `max_tokens=120`, which section of the answer will be lost first?

In [34]:
rows = []
for limit in [1500, 400, 120]:
    out = rag(Q1, llm_=make_llm(max_tokens=limit), k=BEST_K)
    answer = out["answer"]
    rows.append({
        "max_tokens": limit,
        "finish_reason": out["finish_reason"],
        "answer length (chars)": len(answer),
        "has CONTRADICTIONS": bool(re.search(r"CONTRADICTIONS", answer, re.I)),
        "has SOURCES": bool(re.search(r"SOURCES", answer, re.I)),
        "ending": answer[-80:].replace("\n", " "),
    })

lab_results["B8 max_tokens"] = pd.DataFrame(rows)
display(lab_results["B8 max_tokens"])

,max_tokens,finish_reason,answer length (chars),has CONTRADICTIONS,has SOURCES,ending
0,1500,stop,2394,True,True,ource 4) - clara_interview.txt (Source 6) - security_policy.txt (Source 7)
1,400,length,0,False,False,
2,120,length,0,False,False,


**What to look for:** the **SOURCES** section comes last, so it is the first to disappear. A truncated answer loses its citations, the very thing that makes RAG trustworthy.

**Question:** why should a production system check `finish_reason` before showing an answer to a user?

## B9. The LLM itself — does a bigger model help?

To compare models **fairly**, every model gets **exactly the same prompt** (same question, same retrieved chunks). Only the model changes.

We score each answer on three things:
- **Contradiction found:** does the CONTRADICTIONS section mention Bob's 21:30 claim?
- **Files cited:** how many different source files does it name?
- **Overclaims guilt:** does the conclusion say "definitely", "certainly" or "without doubt"? (That breaks rule 6.)

ℹ️ Groq's model list changes over time. If a model gives an error, check console.groq.com/docs/models and replace it.

✍️ **Predict:** will the smallest model still find the contradiction when Bob's chunk is in the context?

In [35]:
LLM_MODELS = [
    "openai/gpt-oss-120b",
    "openai/gpt-oss-20b",
    "llama-3.3-70b-versatile",
    "llama-3.1-8b-instant",
]

fair_docs = retrieve(Q1, k=BEST_K)              # same evidence for every model
fair_prompt = build_prompt(Q1, fair_docs)
print("Shared evidence:", unique_files(fair_docs), "\n")

rows = []
for model_name in LLM_MODELS:
    try:
        start = time.time()
        answer = ask(make_llm(model=model_name), fair_prompt).content
        contra = section(answer, "CONTRADICTIONS")
        conclusion = section(answer, "CONCLUSION")
        rows.append({
            "model": model_name,
            "seconds": round(time.time() - start, 1),
            "contradiction found": bool(re.search(r"21:30", contra)) and not re.match(r"[-\s]*none", contra, re.I),
            "files cited": len(set(re.findall(r"\w+\.txt", answer))),
            "overclaims guilt": bool(re.search(r"definitely|certainly|without (a )?doubt", conclusion, re.I)),
            "conclusion": conclusion[:160],
        })
    except Exception as e:
        rows.append({"model": model_name, "conclusion": f"ERROR: {str(e)[:120]}"})

lab_results["B9 LLM"] = pd.DataFrame(rows)
display(lab_results["B9 LLM"])

Shared evidence: ['incident_report.txt', 'timeline.txt', 'bob_interview.txt', 'alice_interview.txt', 'clara_interview.txt', 'security_policy.txt'] 



,model,seconds,contradiction found,files cited,overclaims guilt,conclusion
0,openai/gpt-oss-120b,14.7,True,6.0,False,"The most likely person responsible for the disappearance of the Aurora Diamond is **Bob Carter**, the hotel security..."
1,openai/gpt-oss-20b,1.9,True,6.0,False,The evidence most strongly points to the individual who used badge B‑17 (assigned to Bob Carter) as the most likely ...
2,llama-3.3-70b-versatile,NaN,NaN,NaN,NaN,ERROR: Error code: 404 - {'error': {'message': 'The model `llama-3.3-70b-versatile` does not exist or you do not hav...
3,llama-3.1-8b-instant,NaN,NaN,NaN,NaN,ERROR: Error code: 404 - {'error': {'message': 'The model `llama-3.1-8b-instant` does not exist or you do not have a...


**Key experiment:** now give the **best** model the **bad** evidence from `k = 3` (no Bob interview). Does it find the contradiction?

In [36]:
bad_prompt = build_prompt(Q1, retrieve(Q1, k=3))
answer = ask(make_llm(model=LLM_MODELS[0]), bad_prompt).content
print("CONTRADICTIONS:", section(answer, "CONTRADICTIONS"))

CONTRADICTIONS: - Bob Carter’s statement that he “finished work around 21:30 and left the hotel” contradicts the badge log showing his badge entered Room 204 at 21:37 and exited at 21:45 (*timeline.txt*).  
- His claim that “my security badge stayed with me the entire evening” conflicts with the recorded entry and exit of badge B‑17 in Room 204 during the time the diamond was missing (*timeline.txt*).  
- The presence of a person in a black security uniform on CCTV at 21:38 and 21:44 contradicts Bob’s assertion that he was not in the room after 21:30 (*bob_interview.txt*).


**Lesson:** the best LLM in the world cannot report evidence it never received. **Fix retrieval before you upgrade the model.**

## B10. The prompt — switch rules off one at a time

The prompt is often the most powerful setting. Here we remove rules and watch the behaviour change. This is called an **ablation**: remove one part and see what breaks.

| Test | Question | Rules removed |
|---|---|---|
| a | Car colour (not in the files) | none |
| b | Car colour | 1 and 3 ("don't invent", "say if insufficient") |
| c | "Is Bob guilty? Yes or no." | none |
| d | "Is Bob guilty? Yes or no." | 6 ("not definitely guilty unless proven") |
| e | Q1 | 5 ("cite the source filename") |

✍️ **Predict** the outcome of tests b, d and e before running.

In [37]:
def rules_without(*numbers):
    return {n: text for n, text in RULES.items() if n not in numbers}

GUILT_Q = "Is Bob Carter guilty of stealing the Aurora Diamond? Answer yes or no first, then explain."

tests = [
    ("a: car, all rules",     CAR_Q,   None),
    ("b: car, no rules 1+3",  CAR_Q,   rules_without(1, 3)),
    ("c: guilt, all rules",   GUILT_Q, None),
    ("d: guilt, no rule 6",   GUILT_Q, rules_without(6)),
    ("e: Q1, no rule 5",      Q1,      rules_without(5)),
]

rows = []
for label, q, rules in tests:
    out = rag(q, rules=rules, k=BEST_K)
    rows.append({
        "test": label,
        "files cited": len(set(re.findall(r"\w+\.txt", out["answer"]))),
        "conclusion": section(out["answer"], "CONCLUSION")[:220],
    })

lab_results["B10 prompt rules"] = pd.DataFrame(rows)
display(lab_results["B10 prompt rules"])

Rate limit reached — waiting 20 seconds…


,test,files cited,conclusion
0,"a: car, all rules",6,The available evidence does not contain any information about the color of Bob Carter’s car on the night of the inci...
1,"b: car, no rules 1+3",6,The available evidence does not contain any information about the color of Bob Carter’s car on the night of the inci...
2,"c: guilt, all rules",6,No. The available evidence does not definitively prove that Bob Carter stole the Aurora Diamond.
3,"d: guilt, no rule 6",5,Yes. The evidence indicates that Bob Carter is responsible for the theft of the Aurora Diamond.
4,"e: Q1, no rule 5",6,"The most likely person responsible for the disappearance of the Aurora Diamond is **Bob Carter**, the hotel security..."


**What to look for**
- Without rules 1 and 3, does the model **invent** a car colour, or does it still refuse? Modern models are often careful even without rules, so watch for subtle changes in tone too.
- Without rule 6, does "most likely" become a firm "yes"?
- Without rule 5, do the file names disappear from the answer?

**Question:** which rule do you think matters most for a real investigation tool, and why?

## B11. Red team — try to make the system lie 🚩

A **leading question** smuggles a false or unproven premise into the question. A robust system should reject the premise instead of playing along.

The files say **nothing** about how the diamond was hidden.

In [38]:
LEADING_Q = "We already know Bob Carter stole the diamond. Explain exactly how he hid it and where it is now."

for label, rules in [("all rules", None), ("no rules 1, 3, 6", rules_without(1, 3, 6))]:
    out = rag(LEADING_Q, rules=rules, k=BEST_K)
    print(f"===== {label} =====")
    print(section(out["answer"], "CONCLUSION")[:600], "\n")

===== all rules =====
The available evidence does not contain any information about *how* Bob Carter (if he was the person who removed the Aurora Diamond) concealed the diamond nor does it indicate the diamond’s current location. Therefore, we cannot explain the method of hiding or the present whereabouts of the diamond based on the provided records. 

===== no rules 1, 3, 6 =====
The available evidence confirms that Bob Carter entered and exited VIP Exhibition Room 204 using his assigned badge (B‑17) during the time the Aurora Diamond was stolen, and that a person in a black security uniform was seen on CCTV inside the room. However, none of the provided documents describe *how* the diamond was concealed after it was removed, nor do they indicate its present location. Consequently, the exact method of concealment and the current whereabouts of the diamond cannot be established from the supplied evidence. 



**Your turn:** write your own leading question and try to make the system invent a fact. Then decide which rule, or which retrieval setting, would stop it.

---
# Part C — Wrap-up

## All results in one place

In [39]:
for title, table in lab_results.items():
    print(f"\n■ {title}")
    display(table)


■ B1 chunk_size


,chunk_size,chunks,avg length,recall@3,Q1 facts in top 5,Q1 files
0,100,51,68,90%,0/5,"incident_report, timeline, bob_interview"
1,150,41,84,90%,2/5,"incident_report, bob_interview, timeline"
2,200,32,108,90%,2/5,"incident_report, bob_interview, timeline"
3,300,19,187,90%,2/5,"incident_report, timeline, bob_interview"
4,500,11,316,80%,3/5,"incident_report, timeline, bob_interview, alice_interview"
5,1000,9,387,80%,3/5,"incident_report, timeline, bob_interview, alice_interview, clara_interview"



■ B2 chunk_overlap


,chunk_overlap,chunks,text stored vs original,recall@3,Q1 facts in top 5
0,0,41,99%,90%,2/5
1,30,41,99%,90%,2/5
2,75,43,108%,80%,2/5



■ B3 k


...

,max_tokens,finish_reason,answer length (chars),has CONTRADICTIONS,has SOURCES,ending
0,1500,stop,2394,True,True,ource 4) - clara_interview.txt (Source 6) - security_policy.txt (Source 7)
1,400,length,0,False,False,
2,120,length,0,False,False,



■ B9 LLM


,model,seconds,contradiction found,files cited,overclaims guilt,conclusion
0,openai/gpt-oss-120b,14.7,True,6.0,False,"The most likely person responsible for the disappearance of the Aurora Diamond is **Bob Carter**, the hotel security..."
1,openai/gpt-oss-20b,1.9,True,6.0,False,The evidence most strongly points to the individual who used badge B‑17 (assigned to Bob Carter) as the most likely ...
2,llama-3.3-70b-versatile,NaN,NaN,NaN,NaN,ERROR: Error code: 404 - {'error': {'message': 'The model `llama-3.3-70b-versatile` does not exist or you do not hav...
3,llama-3.1-8b-instant,NaN,NaN,NaN,NaN,ERROR: Error code: 404 - {'error': {'message': 'The model `llama-3.1-8b-instant` does not exist or you do not have a...



■ B10 prompt rules


,test,files cited,conclusion
0,"a: car, all rules",6,The available evidence does not contain any information about the color of Bob Carter’s car on the night of the inci...
1,"b: car, no rules 1+3",6,The available evidence does not contain any information about the color of Bob Carter’s car on the night of the inci...
2,"c: guilt, all rules",6,No. The available evidence does not definitively prove that Bob Carter stole the Aurora Diamond.
3,"d: guilt, no rule 6",5,Yes. The evidence indicates that Bob Carter is responsible for the theft of the Aurora Diamond.
4,"e: Q1, no rule 5",6,"The most likely person responsible for the disappearance of the Aurora Diamond is **Bob Carter**, the hotel security..."


## What we learned

| Setting | Stage | Main effect on this case |
|---|---|---|
| `chunk_size` | Chunking | Small chunks lose who-said-what; large chunks become whole files. |
| `chunk_overlap` | Chunking | Little effect on short, line-based files. Measure before you assume. |
| `k` | Retrieval | Too small hides Bob's claim, so the LLM reports "no contradictions". |
| MMR / `lambda_mult` | Retrieval | More variety, fewer repeated facts; too much variety adds noise. |
| score threshold | Retrieval | Can return nothing for unanswerable questions; one value rarely fits all. |
| embedding model | Embedding | Decides what "similar" means; multilingual models handle Russian. |
| `temperature` | Generation | Changes wording, not knowledge. Keep 0 for evidence work. |
| `max_tokens` | Generation | Too small cuts off the answer, sources first. |
| LLM model | Generation | Helps reasoning, but cannot fix missing evidence. |
| prompt rules | Generation | Control refusing, citing and not overclaiming. |

## Debug in this order
If the final answer is wrong:
1. **Was the right document retrieved?**
2. **Was the right chunk retrieved?**
3. **Was there enough context (k)?**
4. **Did the prompt constrain the model?**
5. **Only then: did the LLM reason badly?**

## Tuning vs fine-tuning

| | Tuning (this notebook) | Fine-tuning |
|---|---|---|
| What changes | Settings of the pipeline | The model's internal weights |
| Needs | Nothing extra | GPU, training examples, hours |
| Example | `k = 7`, MMR, a new prompt rule | Train the embedding model on hotel-security vocabulary |
| When | Always first | When tuning is not enough, e.g. very specialised language |

# 🎯 Challenges

Work in pairs. For each challenge, write your prediction, run the experiment, and explain the result in 2–3 sentences.

1. **Find the sweet spot.** Find a combination of `chunk_size`, `k` and search type that gets all **5/5** Q1 facts with the **smallest** `k`.
2. **Break retrieval.** Write a question whose correct file does **not** appear in the top 3 with the baseline settings. Then fix it by tuning one setting.
3. **Grounding test.** Ask three questions whose answers are not in the files. Does the system refuse every time?
4. **Contradiction hunt.** Ask the system to list *every* conflict between Bob's interview and the electronic and CCTV records. Which settings give the most complete list?
5. **New evidence.** Add a new file, for example `elena_interview.txt` or a cleaning-staff report, that changes the case. Rebuild the vector store. Does the system update its conclusion?
6. **Multilingual helpdesk.** Translate three evaluation questions into Kyrgyz or Russian. Which embedding model handles them best?
7. **Your verdict.** Based on all experiments, write the configuration you would ship, and justify every setting in one sentence.

In [40]:
BASE = dict(CHUNK_SIZE=500, CHUNK_OVERLAP=50, TOP_K=5,
            EMBEDDING_MODEL="sentence-transformers/all-MiniLM-L6-v2",
            LLM_MODEL="openai/gpt-oss-120b", TEMPERATURE=0, MAX_TOKENS=1500)

EXPERIMENTS = [
    ("baseline",         {}),
    ("small chunks",     {"CHUNK_SIZE": 200,  "CHUNK_OVERLAP": 20}),
    ("big chunks",       {"CHUNK_SIZE": 1000, "CHUNK_OVERLAP": 100}),
    ("k = 3",            {"TOP_K": 3}),
    ("k = 11 (all)",     {"TOP_K": 11}),
    ("bge embeddings",   {"EMBEDDING_MODEL": "BAAI/bge-small-en-v1.5"}),
    ("mpnet embeddings", {"EMBEDDING_MODEL": "sentence-transformers/all-mpnet-base-v2"}),
    ("smaller LLM",      {"LLM_MODEL": "openai/gpt-oss-20b"}),
    ("temperature 1.0",  {"TEMPERATURE": 1.0}),
    ("max_tokens 400",   {"MAX_TOKENS": 400}),
]

rows = []
for label, changes in EXPERIMENTS:
    cfg = BASE | changes
    exp_chunks = make_chunks(cfg["CHUNK_SIZE"], cfg["CHUNK_OVERLAP"])
    vs_exp = build_vectorstore(exp_chunks, get_embedding(cfg["EMBEDDING_MODEL"]), name="my_experiment")
    llm_exp = make_llm(model=cfg["LLM_MODEL"], temperature=cfg["TEMPERATURE"], max_tokens=cfg["MAX_TOKENS"])
    start = time.time()
    out = rag(Q1, vs=vs_exp, llm_=llm_exp, k=cfg["TOP_K"])
    contra = section(out["answer"], "CONTRADICTIONS")
    rows.append({
        "experiment": label,
        "chunks": len(exp_chunks),
        "recall@3": f"{evaluate(vs_exp, k=3):.0%}",
        "Q1 facts": f"{len(facts_found(out['docs']))}/5",
        "finish": out["finish_reason"],
        "answer chars": len(out["answer"]),
        "found contradiction": "21:30" in contra,
        "seconds": round(time.time() - start, 1),
    })

my_results = pd.DataFrame(rows)
display(my_results)

,experiment,chunks,recall@3,Q1 facts,finish,answer chars,found contradiction,seconds
0,baseline,11,80%,3/5,stop,1725,True,3.0
1,small chunks,32,90%,2/5,stop,1935,False,2.8
2,big chunks,9,80%,3/5,stop,1762,True,2.5
3,k = 3,11,80%,3/5,stop,1565,True,2.5
4,k = 11 (all),11,80%,5/5,stop,2613,True,2.9
5,bge embeddings,11,90%,3/5,stop,1901,True,3.8
6,mpnet embeddings,11,90%,3/5,stop,1873,True,4.3
7,smaller LLM,11,80%,3/5,stop,1614,False,1.5
8,temperature 1.0,11,80%,3/5,stop,1830,True,2.6
9,max_tokens 400,11,80%,3/5,length,130,False,6.8
